In [2]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd

from src.extractor import process_company, extract_text_by_page
from src.calculation_engine import generate_financial_analysis, build_financial_summary



In [3]:
PDF_PATH = "../data/Microsoft/microsoft_23'-10k.pdf"
COMPANY  = "microsoft"
YEAR     = 2023
SAVE_DIR = "../data/Microsoft"

# Leave as None for now — we will set this in the next cell after previewing pages
TARGET_PAGES = None

print(f"Config: {COMPANY} {YEAR}")


Config: microsoft 2023


In [27]:
PREVIEW_PAGE = 58   # change this number and re-run

pages = extract_text_by_page(PDF_PATH)
print(f"Total pages in PDF: {len(pages)}")
print(f"\n--- Page {PREVIEW_PAGE} ---\n")
print(pages[PREVIEW_PAGE - 1]["text"][:3000])


  Opened PDF: microsoft_23'-10k.pdf (116 pages total)
Total pages in PDF: 116

--- Page 58 ---

PART II
Item 8
 
ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA
INCOME STATEMENTS
 
(In millions, except per share amounts)
  
   
   
 
 
 
 
 
 
 
 
Year Ended June 30,
 
2023   
2022   
2021  
 
 
 
 
Revenue:
  
    
    
  
Product
 $
64,699   $
72,732   $
71,074  
Service and other
  
147,216    
125,538    
97,014  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Total revenue
  
211,915    
198,270    
168,088  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Cost of revenue:
  
    
    
  
Product
  
17,804    
19,064    
18,219  
Service and other
  
48,059    
43,586    
34,013  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Total cost of revenue
  
65,863    
62,650    
52,232  
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Gross margin
  
146,052    
135,620    
115,856  
Research and development
  
27,195    
24,512    
20,716  
Sales and marketing
  
22,759    
21,825    
20,117  
General and admin

In [4]:
# Update with the actual page numbers you found above
TARGET_PAGES = [58,60]  # example — replace with real ones

metrics = process_company(
    company_name=COMPANY,
    pdf_path=PDF_PATH,
    year=YEAR,
    target_pages=TARGET_PAGES,
    save_dir=SAVE_DIR,
)

print("\nExtracted metrics:")
for key, val in metrics.items():
    yr_val = val["values"].get(str(YEAR), "N/A")
    if isinstance(yr_val, float):
        print(f"  {key:<40} {yr_val:>12,.0f} M")
    else:
        print(f"  {key:<40} {yr_val}")



  Company : MICROSOFT
  PDF     : microsoft_23'-10k.pdf
  Year    : 2023  |  Columns: [2023, 2022, 2021]
  Pages   : [58, 60]

[Step 1/4] Extracting financial lines from PDF...
  Opened PDF: microsoft_23'-10k.pdf (116 pages total)
  Extracted 51 candidate financial lines (26 rows skipped)

[Step 2/4] Assigning fiscal years to numeric columns...
  Year mapping complete: 51 rows with 3-year values

[Step 3/4] Saving raw extraction CSV for inspection...
  [Debug CSV saved] -> ..\data\Microsoft\microsoft_2023_raw_extraction.csv

[Step 4/4] Normalizing labels using mapping config...

Normalizing 51 rows for 'microsoft'...
  [exact]  'total revenue' -> 'total_revenue'
  [exact]  'gross margin' -> 'gross_profit'
  [exact]  'research and development' -> 'research_and_development'
  [exact]  'sales and marketing' -> 'selling_general_administrative'
  [exact]  'operating income' -> 'operating_income'
  [exact]  'net income' -> 'net_income'
  [exact]  'cash and cash equivalents' -> 'cash_and_equ

In [5]:
csv_path = f"../data/Microsoft/microsoft_{YEAR}_raw_extraction.csv"
df = pd.read_csv(csv_path)
print(f"Raw rows extracted: {len(df)}")
df


Raw rows extracted: 51


,label,source_page,2023,2022,2021
0,Year Ended June,58,2023.0,2022.0,2021.0
1,Product,58,64699.0,72732.0,71074.0
2,Service and other,58,147216.0,125538.0,97014.0
3,Total revenue,58,211915.0,198270.0,168088.0
4,Product,58,17804.0,19064.0,18219.0
5,Service and other,58,48059.0,43586.0,34013.0
6,Total cost of revenue,58,65863.0,62650.0,52232.0
7,Gross margin,58,146052.0,135620.0,115856.0
8,Research and development,58,27195.0,24512.0,20716.0
9,Sales and marketing,58,22759.0,21825.0,20117.0


In [8]:
analysis = generate_financial_analysis(YEAR, metrics=metrics)
summary  = build_financial_summary(analysis, company_name="Microsoft")
print(summary)



MICROSOFT FINANCIAL ANALYSIS (2023)

Revenue:
- Revenue: $211,915 million
- Revenue growth: 6.88%

Net Income:
- Net income: $72,361 million
- Net income growth: -0.52%

Profitability:
- Gross margin: 68.92%
- Operating margin: 41.77%
- Net profit margin: 34.15%

Expense Efficiency:
- R&D as % of sales: 12.83%

Balance Sheet:
- Liabilities to assets: 25.28%
- Cash to assets: 8.42%



In [9]:
from src.answer_engine import calculate_answer

# These questions are answered purely by math — no LLM, no guessing
test_questions = [
    "What was Microsoft's net profit margin in 2023?",
    "What was the gross margin % in 2023?",
    "What was the operating margin in 2023?",
    "What was Microsoft's revenue growth in 2023?",
    "What is the R&D as a percentage of sales?",
    "What was the liabilities to assets ratio?",
    "What was the cash to assets ratio?",
]

print("=" * 60)
print("DETERMINISTIC CALCULATION RESULTS — MICROSOFT 2023")
print("=" * 60)

for question in test_questions:
    result = calculate_answer(question, metrics=metrics, year=2023)
    
    if result:
        print(f"\nQ: {question}")
        print(f"A: {result['answer']}")
        print(f"   {result['calculation']}")
    else:
        print(f"\nQ: {question}")
        print(f"A: [No deterministic handler — would go to RAG]")


DETERMINISTIC CALCULATION RESULTS — MICROSOFT 2023

Q: What was Microsoft's net profit margin in 2023?
A: Net profit margin in 2023 was 34.15%.
   Net profit margin = Net income / Total net sales × 100 = $72,361 million / $211,915 million × 100 = 34.15%

Q: What was the gross margin % in 2023?
A: Gross margin percentage in 2023 was 68.92%.
   Gross margin % = Gross margin / Total net sales × 100 = $146,052 million / $211,915 million × 100 = 68.92%

Q: What was the operating margin in 2023?
A: Operating margin in 2023 was 41.77%.
   Operating margin = Operating income / Total net sales × 100 = $88,523 million / $211,915 million × 100 = 41.77%

Q: What was Microsoft's revenue growth in 2023?
A: Total net sales changed by 6.88% in 2023 compared with 2022.
   Revenue growth = (2023 net sales - 2022 net sales) / 2022 net sales × 100 = ($211,915 million - $198,270 million) / $198,270 million × 100 = 6.88%

Q: What is the R&D as a percentage of sales?
A: R&D expense as a percentage of sales i

In [10]:
from src.answer_engine import generate_llm_financial_interpretation

# Pass the deterministic summary to the LLM
# The LLM interprets — it does NOT calculate
# This lets you catch hallucinations: any number it mentions must be in the summary

print("Sending summary to LLM for interpretation...\n")
interpretation = generate_llm_financial_interpretation(summary)
print(interpretation)


Sending summary to LLM for interpretation...

 In the provided financial analysis summary for Microsoft in 2023, several key observations can be made regarding profitability, growth, efficiency, and financial health:

1. Profitability: The company demonstrates strong profitability with a gross margin of 68.92%, operating margin of 41.77%, and net profit margin of 34.15%. These figures suggest that Microsoft generates substantial profits from its sales, operational activities, and overall business operations.

2. Growth: While the revenue growth rate is positive at 6.88%, the net income growth rate shows a slight decrease of -0.52%. This discrepancy between revenue and net income growth may indicate potential challenges in managing expenses or increasing profitability despite revenue growth.

3. Efficiency: Microsoft invests 12.83% of its sales into research and development (R&D), which is a significant investment in innovation and future product development. This high R&D expenditure a

In [11]:
import ollama, json
from src.answer_engine import calculate_answer
from src.utils import format_money, format_percent

question = "What was Microsoft's net profit margin in 2023 and what does it indicate?"

# Step 1: get the deterministic answer
calc = calculate_answer(question, metrics=metrics, year=2023)

# Step 2: build a grounded prompt — LLM can only interpret, not invent numbers
prompt = f"""
You are a financial analyst assistant.

A user asked: "{question}"

The verified calculation result is:
{json.dumps(calc, indent=2)}

Rules you must follow:
1. Use ONLY the numbers from the calculation result above.
2. Do NOT invent or estimate any other numbers.
3. Interpret what this margin means for Microsoft's business quality.
4. Keep your answer to 3-4 sentences.

Answer:
"""

response = ollama.chat(
    model="mistral",
    messages=[{"role": "user", "content": prompt}],
    options={"temperature": 0.2},
)

print("Question:", question)
print()
print("Deterministic answer:", calc["answer"])
print()
print("LLM Interpretation:")
print(response["message"]["content"])


Question: What was Microsoft's net profit margin in 2023 and what does it indicate?

Deterministic answer: Net profit margin in 2023 was 34.15%.

LLM Interpretation:
 The net profit margin of Microsoft in 2023 was 34.15%, indicating a high level of profitability for the company. This suggests that for every dollar of revenue generated, Microsoft kept $0.34 as net profit. A high net profit margin is generally considered a positive sign, demonstrating Microsoft's ability to efficiently manage its costs and generate substantial profits compared to its sales.


In [13]:
# Quick sanity check: verify the numbers the LLM mentioned match our metrics

ground_truth = {
    "total_revenue_2023":       metrics["total_revenue"]["values"]["2023"],
    "net_income_2023":          metrics["net_income"]["values"]["2023"],
    "gross_profit_2023":        metrics["gross_profit"]["values"]["2023"],
    "operating_income_2023":    metrics["operating_income"]["values"]["2023"],
    "total_assets_2023":        metrics["total_assets"]["values"]["2023"],
    "total_liabilities_2023":   metrics["total_liabilities"]["values"]["2023"],
    "cash_and_equivalents_2023":metrics["cash_and_equivalents"]["values"]["2023"],
}

print("GROUND TRUTH — MICROSOFT 2023 (in millions USD)")
print("-" * 45)
for key, val in ground_truth.items():
    print(f"  {key:<35} ${val:>10,.0f} M")

print()
print("Cross-check any number the LLM mentioned above against this table.")
print("If it invented a number not here -> hallucination detected.")


GROUND TRUTH — MICROSOFT 2023 (in millions USD)
---------------------------------------------
  total_revenue_2023                  $   211,915 M
  net_income_2023                     $    72,361 M
  gross_profit_2023                   $   146,052 M
  operating_income_2023               $    88,523 M
  total_assets_2023                   $   411,976 M
  total_liabilities_2023              $   104,149 M
  cash_and_equivalents_2023           $    34,704 M

Cross-check any number the LLM mentioned above against this table.
If it invented a number not here -> hallucination detected.
